# Feature Engineering & the Snowflake Feature Store

**Session 2 -- Deep Dive: Feature Engineering, Feature Stores & Experiment Tracking**  
**Notebook 1 of 4** | ~15 minutes

---

## What You Will Learn

| Concept | What We Cover |
|---------|---------------|
| **Feature Store** | Initialize a centralized feature store inside a Snowflake schema |
| **Entities** | Define the business objects your features describe |
| **Feature Views** | Register reusable, auto-refreshing feature pipelines |
| **Feature Descriptions** | Attach human-readable metadata for discoverability |
| **Managed Refresh** | Let Snowflake keep features fresh via Dynamic Tables |

---

## Why a Feature Store?

In traditional ML workflows, every team builds their own feature pipelines -- leading to:

- **Duplicated effort** -- the same "pulse pressure" feature computed three different ways
- **Training-serving skew** -- features computed differently at training time vs. inference time
- **No discoverability** -- data scientists cannot find features that already exist
- **No lineage** -- impossible to know which models depend on which features

The Snowflake Feature Store solves all of these by providing a **single source of truth** for features,
backed by Dynamic Tables that automatically refresh on a schedule you control.

### Key Capabilities

| Capability | How Snowflake Does It |
|------------|----------------------|
| Storage | Dynamic Tables with automatic refresh |
| Refresh | Built-in `refresh_freq` — declarative SQL scheduler |
| Online Serving | Native Online Feature Tables (Hybrid Tables) |
| Point-in-Time Joins | `fs.generate_dataset()` with immutable Datasets |
| Lineage | Automatic: Table → FeatureView → Dataset → Model |
| Governance | Snowflake RBAC + Tags |

---

## Use Case: Patient Risk Stratification

We are building a **patient risk classifier** for a hospital system. Raw EMR (Electronic Medical Record)
data flows into `RAW_PATIENT_DATA` and contains vitals, labs, demographics, and clinical context.

We will engineer **four domain-driven features** on top of the raw data:

| Feature | Formula | Clinical Meaning |
|---------|---------|------------------|
| `SHOCK_INDEX` | Heart Rate / Systolic BP | Hemodynamic instability indicator |
| `PULSE_PRESSURE` | Systolic BP - Diastolic BP | Cardiovascular health proxy |
| `BMI_CATEGORY` | Categorical bucket of BMI | Weight classification |
| `VITAL_SIGNS_SEVERITY` | Composite score 0-9 | Multi-organ severity aggregate |

These features will be registered once and reused across **every model and team** that needs them.

## 1 | Environment Setup

We import the shared configuration so that database names, schema names,
and feature lists are defined in exactly one place.

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
import logging

from utils import get_config
from utils import get_session

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
config = get_config("config.yaml")

DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
WH = config.snowflake.warehouse

# == Create Snowflake Session ==
session = get_session(
    connection_name=config.snowflake.connection_name,
    database_name=DB,
    schema_name=SCHEMA,
    warehouse_name=WH,
)

print(f"Connected as {session.get_current_user()} | Role: {session.get_current_role()}")
print(f"Context: {DB}.{SCHEMA} | Warehouse: {WH}")

## 2 | Explore the Raw Data

Before engineering features, let's understand what we are working with.
`RAW_PATIENT_DATA` contains ~10,000 synthetic EMR records with demographics,
vitals, labs, and a physician-assigned `RISK_LEVEL` label.

In [ ]:
raw_table = config.full_raw_table
raw_df = session.table(raw_table)

print(f"Source table: {raw_table}")
print(f"Row count:   {raw_df.count():,}")
print(f"Columns:     {len(raw_df.columns)}")
print()
raw_df.select(
    "PATIENT_ID",
    "TIMESTAMP",
    "AGE",
    "HEART_RATE",
    "SYSTOLIC_BP",
    "DIASTOLIC_BP",
    "OXYGEN_SATURATION",
    "BMI",
    "RISK_LEVEL",
).show(5)

In [ ]:
print("Risk Level Distribution:")
raw_df.group_by("RISK_LEVEL").count().sort("RISK_LEVEL").show()

## 3 | Initialize the Feature Store

### Architecture

```
 Snowflake Account
 +---------------------------------------+
 | Database: ML_DEMO_PIPELINE_DB         |
 |  +-----------------------------------+|
 |  | Schema: HEALTHCARE (Feature Store)||
 |  |                                   ||
 |  |  [Entity Tags]                    ||
 |  |    PATIENT (join_key=PATIENT_ID)  ||
 |  |                                   ||
 |  |  [Dynamic Tables]                 ||
 |  |    PATIENT_FEATURES$v1            ||
 |  |      auto-refresh every 1 min     ||
 |  |      from RAW_PATIENT_DATA        ||
 |  +-----------------------------------+|
 +---------------------------------------+
```

In [ ]:
from snowflake.ml.feature_store import CreationMode, Entity, FeatureStore, FeatureView

fs = FeatureStore(
    session=session,
    database=DB,
    name=SCHEMA,
    default_warehouse=WH,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

print(f"Feature Store initialized: {DB}.{SCHEMA}")

## 4 | Register an Entity

An **Entity** represents the real-world object that features describe.
Think of it as the primary key (or composite key) that connects all feature views for a given subject.

In our case, every feature is about a **patient encounter**, keyed by `PATIENT_ID`.

### Why Entities Matter

- **Feature discovery** -- "Show me all features registered for the PATIENT entity"
- **Automatic joins** -- the Feature Store uses entity join keys to stitch features together
- **Governance** -- entities are implemented as Snowflake tags, which means they integrate with RBAC

### Design Benefit

Entities are first-class objects that can be shared across multiple Feature Views,
enabling consistent join keys and discoverability across teams.

In [ ]:
patient_entity = Entity(
    name="PATIENT",
    join_keys=["PATIENT_ID"],
    desc="Hospital patient identified by PATIENT_ID. Each row represents a single encounter.",
)

fs.register_entity(patient_entity)


print("Entity PATIENT registered.")
print(f"  Join keys:    {patient_entity.join_keys}")
print(f"  Description:  {patient_entity.desc}")

### 4.1 List available entities in the Feature Store

In [ ]:
print("All entities in the Feature Store:")
fs.list_entities().to_pandas()

## 5 | Feature Engineering with Snowpark

Feature engineering happens as **lazy Snowpark transformations** -- the computations are expressed
as a DataFrame but executed entirely within Snowflake's query engine.  No data leaves the platform.

### The Four Engineered Features

| # | Feature | Logic | Clinical Rationale |
|---|---------|-------|-----------|
| 1 | `SHOCK_INDEX` | `HEART_RATE / SYSTOLIC_BP` | Values > 0.9 indicate shock; widely used in emergency medicine |
| 2 | `PULSE_PRESSURE` | `SYSTOLIC_BP - DIASTOLIC_BP` | Narrow pulse pressure suggests reduced cardiac output |
| 3 | `BMI_CATEGORY` | Categorical bucket of BMI | Converts continuous BMI into clinically meaningful groups |
| 4 | `VITAL_SIGNS_SEVERITY` | Sum of abnormal-vital flags (0-9) | A multi-organ early-warning score |

Notice how these encode **domain knowledge** that a raw model might struggle to learn on its own.
By placing them in the Feature Store, every downstream model benefits automatically.

### Key Insight: Snowpark = Server-Side Execution

Unlike pandas (which pulls data to the client), Snowpark transformations push computation
to Snowflake's engine. The DataFrame below is just a **query plan** until we call `.show()` or
register it as a Feature View.

In [ ]:
import snowflake.snowpark.functions as F

source_df = session.table(raw_table)

feature_df = (
    source_df.with_column("SHOCK_INDEX", F.col("HEART_RATE") / F.col("SYSTOLIC_BP"))
    .with_column("PULSE_PRESSURE", F.col("SYSTOLIC_BP") - F.col("DIASTOLIC_BP"))
    .with_column(
        "BMI_CATEGORY",
        F.when(F.col("BMI") < 18.5, F.lit("UNDERWEIGHT"))
        .when(F.col("BMI") < 25.0, F.lit("NORMAL"))
        .when(F.col("BMI") < 30.0, F.lit("OVERWEIGHT"))
        .otherwise(F.lit("OBESE")),
    )
    .with_column(
        "VITAL_SIGNS_SEVERITY",
        F.when((F.col("HEART_RATE") > 100) | (F.col("HEART_RATE") < 50), F.lit(1)).otherwise(
            F.lit(0)
        )
        + F.when((F.col("SYSTOLIC_BP") > 180) | (F.col("SYSTOLIC_BP") < 90), F.lit(2)).otherwise(
            F.lit(0)
        )
        + F.when(F.col("OXYGEN_SATURATION") < 92, F.lit(2)).otherwise(
            F.when(F.col("OXYGEN_SATURATION") < 95, F.lit(1)).otherwise(F.lit(0))
        )
        + F.when(F.col("RESPIRATORY_RATE") > 24, F.lit(1)).otherwise(F.lit(0))
        + F.when((F.col("TEMPERATURE") > 38.5) | (F.col("TEMPERATURE") < 36.0), F.lit(1)).otherwise(
            F.lit(0)
        ),
    )
)

print("Engineered features preview:")
feature_df.select(
    "PATIENT_ID",
    "TIMESTAMP",
    "SHOCK_INDEX",
    "PULSE_PRESSURE",
    "BMI_CATEGORY",
    "VITAL_SIGNS_SEVERITY",
    "RISK_LEVEL",
).show(5)

### 5.1 View the query plan

In [ ]:
feature_df.explain()

## 6 | Register a Feature View

A **Feature View** wraps the Snowpark transformation we just wrote and materializes it as a
Snowflake **Dynamic Table** that auto-refreshes on the schedule we specify.

#### Key Parameters

| Parameter | Value | Purpose |
|-----------|-------|-----|
| `entities` | `[patient_entity]` | Links features to the PATIENT join key |
| `feature_df` | Our Snowpark DataFrame | Defines the transformation logic |
| `timestamp_col` | `"TIMESTAMP"` | Enables point-in-time correct retrieval (notebook 2) |
| `refresh_freq` | `"1 minute"` | How often the Dynamic Table refreshes |
| `desc` | Human-readable text | Discoverability for other teams |


#### Key Benefit

The Feature View *is* the pipeline — the refresh is declarative and fully managed.
No orchestration code, no external schedulers, no manual writes required.

### 6.1 Initialize the Feature View

In [ ]:
fv = FeatureView(
    name="PATIENT_FEATURES",
    entities=[patient_entity],
    feature_df=feature_df,
    timestamp_col="TIMESTAMP",
    refresh_freq="1 minute",
    desc=(
        "Raw vitals, labs, and demographics plus 4 engineered features: "
        "SHOCK_INDEX, PULSE_PRESSURE, BMI_CATEGORY, VITAL_SIGNS_SEVERITY. "
        "Refreshes every 1 minute from RAW_PATIENT_DATA."
    ),
)

### 6.2 Attach Feature-Level Descriptions

Feature descriptions make the store **self-documenting**. When another data scientist queries
`fs.list_feature_views()`, they can see exactly what each column means without reading code.

This is the Snowflake equivalent of a data dictionary -- but it lives right next to the data.

In [ ]:
fv.attach_feature_desc(
    {
        "SHOCK_INDEX": "Heart Rate / Systolic BP -- values > 0.9 suggest hemodynamic instability",
        "PULSE_PRESSURE": "Systolic BP minus Diastolic BP -- narrow values indicate reduced cardiac output",
        "BMI_CATEGORY": "Weight classification: UNDERWEIGHT / NORMAL / OVERWEIGHT / OBESE",
        "VITAL_SIGNS_SEVERITY": "Composite severity score 0-9 based on abnormal vital sign thresholds",
        "HEART_RATE": "Resting heart rate in beats per minute",
        "SYSTOLIC_BP": "Systolic blood pressure in mmHg",
        "DIASTOLIC_BP": "Diastolic blood pressure in mmHg",
        "OXYGEN_SATURATION": "SpO2 percentage -- below 92% is critical",
        "TEMPERATURE": "Body temperature in Celsius",
        "RESPIRATORY_RATE": "Breaths per minute",
        "GLUCOSE_LEVEL": "Blood glucose in mg/dL",
        "CREATININE": "Serum creatinine in mg/dL -- kidney function marker",
        "HEMOGLOBIN": "Hemoglobin in g/dL -- oxygen-carrying capacity",
        "WBC_COUNT": "White blood cell count in thousands/uL -- infection marker",
        "AGE": "Patient age in years",
        "BMI": "Body Mass Index (kg/m2)",
        "RISK_LEVEL": "Physician-assigned risk: LOW / MEDIUM / HIGH / CRITICAL",
        "MEDICATION_COUNT": "Number of active medications at time of encounter",
        "COMORBIDITY_COUNT": "Number of concurrent chronic conditions diagnosed",
        "ADMISSION_TYPE": "Hospital admission category: EMERGENCY / URGENT / ELECTIVE / NEWBORN",
        "INSURANCE_TYPE": "Patient insurance coverage: PRIVATE / MEDICARE / MEDICAID / SELF_PAY",
        "PREVIOUS_ADMISSIONS": "Total prior hospital admissions in patient history",
        "GENDER": "Patient biological sex: M / F",
        "PRIMARY_DIAGNOSIS": "Patient primary diagnosed condition",
    }
)

print("Feature descriptions attached.")

### 6.3 Register the Feature View

#### What Happens Under the Hood

The `register_feature_view` function performs the following SQL command under the hood.

```sql
CREATE DYNAMIC TABLE PATIENT_FEATURES$v1
  TARGET_LAG = '1 minute'
  WAREHOUSE = ML_DEMO_WAREHOUSE
  AS SELECT ... (your Snowpark SQL)
```

In [ ]:
registered_fv = fs.register_feature_view(
    feature_view=fv,
    version="v1",
    block=True,
    overwrite=True,
)

print(f"Feature View registered: {registered_fv.name}/{registered_fv.version}")
print(f"Status:        {registered_fv.status}")
print(f"Refresh freq:  {registered_fv.refresh_freq}")
print(f"Feature count: {len(registered_fv.feature_names)}")

## 7 | Inspect the Feature Store

Let's verify everything was registered correctly and explore the metadata.
This is what another team member would see when they discover your features.

In [ ]:
print("=" * 60)
print("ENTITIES")
print("=" * 60)
display(fs.list_entities().to_pandas())


print("=" * 60)
print("FEATURE VIEWS")
print("=" * 60)
fv_list = fs.list_feature_views().to_pandas()
display(fv_list[["NAME", "VERSION", "DESC", "REFRESH_FREQ", "SCHEDULING_STATE"]])


print("=" * 60)
print("FEATURE NAMES & DESCRIPTIONS")
print("=" * 60)
for name, desc in registered_fv.feature_descs.items():
    print(f"  {str(name):30s} -> {str(desc)}")

## 8 | Read Feature Values (Offline Store)

The `read_feature_view` method reads directly from the materialized Dynamic Table.
This is the **offline store** -- optimized for batch reads like training and analytics.

| Store Type | Backed By | Latency | Use Case |
|------------|-----------|---------|----------|
| **Offline** | Dynamic Table | Seconds | Training, batch inference, analytics |
| **Online** | Hybrid Table | Milliseconds | Real-time serving (covered in notebook 2) |

In [ ]:
sample = fs.read_feature_view(registered_fv)

print(f"Feature View contains {sample.count():,} rows")
sample.select(
    "PATIENT_ID",
    "HEART_RATE",
    "SYSTOLIC_BP",
    "SHOCK_INDEX",
    "PULSE_PRESSURE",
    "BMI_CATEGORY",
    "VITAL_SIGNS_SEVERITY",
    "RISK_LEVEL",
).show(10)

## 9 | Observe Automatic Refresh

Because the Feature View is backed by a Dynamic Table with `refresh_freq = '1 minute'`,
Snowflake automatically detects changes in `RAW_PATIENT_DATA` and re-computes the features.

The refresh is **incremental** when possible -- only changed rows are reprocessed.

We can inspect the refresh history to see when the last refresh occurred and how long it took.
This is fully managed and requires zero orchestration code.

In [ ]:
refresh_history = fs.get_refresh_history(registered_fv)
display(refresh_history.to_pandas().head(15))

## 10 | Summary

In this notebook we:

1. **Initialized** a Feature Store inside an existing Snowflake schema
2. **Registered** a `PATIENT` entity with `PATIENT_ID` as the join key
3. **Engineered** four clinical features using Snowpark lazy transformations
4. **Attached** human-readable descriptions for discoverability
5. **Registered** a `PATIENT_FEATURES` Feature View backed by a Dynamic Table
6. **Inspected** the metadata, underlying SQL, and refresh history

### Key Takeaways

| Takeaway | Detail |
|----------|--------|
| **Zero infrastructure** | Feature Store = Snowflake schema. No external services. |
| **Auto-refresh** | Dynamic Tables handle scheduling, change detection, and incremental refresh. |
| **Discoverability** | Feature descriptions and entity tags make features findable by any team. |
| **Compute stays in Snowflake** | All transformations run server-side via Snowpark. |
| **Idempotent** | Re-running this notebook does not break anything (`overwrite=True`). |
| **No training-serving skew** | The same Feature View definition serves both training and inference. |

### What Comes Next

```
  [Notebook 1]            [Notebook 2]            [Notebook 3]            [Notebook 4]
  Feature Engineering --> Advanced Feature Store -> Experiment Tracking -> Model Registry
  (this notebook)         Point-in-time datasets   Baseline + variations   Register, version,
                          Online serving            Compare runs            compare, promote
                          Sharing & lineage
```

---

**Next -->** [02_feature_store_advanced.ipynb](02_feature_store_advanced.ipynb) -- Point-in-time datasets, online serving, sharing, and lineage